# Baseline B0 — cross-dataset waste classification (2× T4)

**Chuẩn bị trước khi chạy**
1. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **On**.
2. Nén thư mục `datasets` thành **một file `.zip`**, upload lên Google Drive, chia sẻ *Anyone with the link*, rồi dán link vào `DRIVE_URL` ở cell dưới.
   (Không dùng link *folder*: gdown chỉ tải được tối đa 50 file mỗi folder.)
3. *Run All*.

**Luồng chạy:** tải data → `prepare.py` (dedup, manifest) → `extract.py` song song trên 2 GPU → `probe.py` song song trên 4 CPU → bảng kết quả → nén outputs.
Các bước đều *resume* được: chạy lại notebook sẽ bỏ qua phần đã xong.

In [ ]:
# ===== CẤU HÌNH =====
DRIVE_URL = "PASTE_GOOGLE_DRIVE_ZIP_LINK_HERE"   # <-- cập nhật link Google Drive (.zip)
SEED      = 0                                     # seed điều khiển cách chia train/val/test

REPO_URL  = "https://github.com/trong5nhan6/trash.git"
REPO_DIR  = "/kaggle/working/trash"
DATA_DIR  = "/tmp/data"            # nơi giải nén (ổ /tmp lớn, không tính vào 20GB output)
ROOT      = "/tmp/root"            # thư mục chuẩn hoá tên 4 dataset (symlink)
OUT       = "/kaggle/working/outputs"

# Chia backbone cho 2 GPU, cân bằng theo khối lượng tính (DINOv2-L nặng nhất)
GPU_GROUPS = [
    ["dinov2_l", "resnet50", "effnet_b0"],
    ["dinov2_b", "clip_b16", "siglip_b16", "convnext_t"],
]

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, os
print("CUDA devices:", torch.cuda.device_count(), "| CPU cores:", os.cpu_count())

In [ ]:
# Lấy code mới nhất + thư viện
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
!pip install -q gdown "timm>=1.0.9"
!git -C {REPO_DIR} log --oneline -1

In [ ]:
# Tải và giải nén dữ liệu từ Google Drive (bỏ qua nếu đã có)
import glob, shutil
os.makedirs(DATA_DIR, exist_ok=True)
already = glob.glob(f"{DATA_DIR}/**/TrashBox_train_set", recursive=True)
if not already:
    assert "PASTE_" not in DRIVE_URL, "Hãy điền DRIVE_URL ở cell cấu hình"
    zip_path = f"{DATA_DIR}/data.zip"
    subprocess.run(["gdown", "--fuzzy", DRIVE_URL, "-O", zip_path], check=True)
    print("Unzipping ..."); shutil.unpack_archive(zip_path, DATA_DIR); os.remove(zip_path)
!du -sh {DATA_DIR}

In [ ]:
# Chuẩn hoá tên thư mục bằng symlink -> không phụ thuộc tên lúc upload (vd. 'archive (2)')
MARKERS = {                      # dataset -> thư mục con đặc trưng nằm ngay bên trong
    "TrashNet":  "dataset-resized",
    "RealWaste": "realwaste-main",
    "TrashBox":  "TrashBox_train_set",
    "Garbage12": "garbage_classification",
}
os.makedirs(ROOT, exist_ok=True)
for name, marker in MARKERS.items():
    hits = [h for h in glob.glob(f"{DATA_DIR}/**/{marker}", recursive=True) if os.path.isdir(h)]
    assert hits, f"Không tìm thấy '{marker}' ({name}) trong {DATA_DIR}"
    parent = os.path.dirname(sorted(hits, key=len)[0])
    link = f"{ROOT}/{name}"
    if os.path.islink(link): os.remove(link)
    os.symlink(parent, link)
    print(f"{name:10s} -> {parent}")

In [ ]:
# Bước 1: quét, kiểm tra ảnh, loại trùng (md5) -> manifest.csv   (CPU, vài phút)
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/manifest.csv"):
    !cd {REPO_DIR} && python code/prepare.py --root {ROOT} --out {OUT}
else:
    print("manifest.csv đã có -> bỏ qua")
!cat {OUT}/table1_counts.csv

In [ ]:
# Bước 2: trích feature — 2 tiến trình, mỗi tiến trình giữ 1 GPU, chạy song song
import time
n_gpu = torch.cuda.device_count()
groups = GPU_GROUPS if n_gpu >= 2 else [sum(GPU_GROUPS, [])]
workers = max(1, os.cpu_count() // len(groups))     # chia CPU cho các tiến trình đọc ảnh

procs = []
for g, bbs in enumerate(groups):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED="1")
    log = open(f"{OUT}/extract_gpu{g}.log", "a")
    cmd = ["python", "code/extract.py", "--root", ROOT, "--out", OUT,
           "--workers", str(workers), "--batch", "128", "--backbones", *bbs]
    procs.append((g, subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    print(f"GPU{g}: {bbs}")

def tail(path, n=2):
    with open(path) as f: return "".join(f.readlines()[-n:]).rstrip()

t0 = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(60)
    print(f"--- {(time.time()-t0)/60:.0f} min ---")
    for g, p, _ in procs:
        print(f"[GPU{g}] {tail(f'{OUT}/extract_gpu{g}.log')}")
for g, p, log in procs:
    log.close()
    print(f"\n===== GPU{g} log =====\n" + tail(f"{OUT}/extract_gpu{g}.log", 12))
    assert p.returncode == 0, f"GPU{g} lỗi (exit {p.returncode}) — xem {OUT}/extract_gpu{g}.log"
!ls -lh {OUT}/feats

In [ ]:
# Bước 3: linear probe (CPU) — 4 backbone song song; 1 seed
!cd {REPO_DIR} && python code/probe.py --out {OUT} --seed {SEED} --jobs {os.cpu_count()}

In [ ]:
# Kết quả: Table 2a + ma trận 4x4 (macro-F1, hàng = train, cột = test)
import pandas as pd
res = f"{OUT}/results/seed{SEED}"
display(pd.read_csv(f"{res}/table2a_summary.csv").sort_values("gap"))
for f in sorted(glob.glob(f"{res}/matrix_macro_f1_*.csv")):
    print(os.path.basename(f)); display(pd.read_csv(f, index_col=0))

In [ ]:
# Nén outputs để tải về (gồm feature .npz để chạy seed khác không cần GPU)
shutil.make_archive(f"/kaggle/working/outputs_seed{SEED}", "zip", OUT)
!ls -lh /kaggle/working/*.zip

### Chạy thêm seed (không cần GPU)
Feature đã có, chỉ cần probe lại với seed khác:
```python
for s in [1, 2]:
    !cd {REPO_DIR} && python code/probe.py --out {OUT} --seed {s} --jobs {os.cpu_count()}
```